# einops-rearrange — faded example 2: Merge attention heads back into a packed tensor

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `einops-rearrange`. Running the beacon reports progress on the `Einops: Rearrange` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Einops: Rearrange` subtopic.
You can copy the token from your Delta Drills account page.

This drill exercises the **atom `einops-rearrange`**, which bridges to the bank subtopic `Einops: Rearrange` for EWMA state. Completing all 5 exercises triggers a single `arena-rating` beacon at the end of the notebook.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "einops-rearrange"
DD_SUBTOPIC = "Einops: Rearrange"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

# Track which exercises passed in this session.
_dd_passed = set()

## Concept

The inverse of splitting heads. After attention you have `(b, h, s, d)` and need `(b, s, h*d)` to feed the output projection. This requires a **reorder** (move `s` ahead of `h`) plus a **composition** `(h d)` on the output side — both expressible in one `rearrange` pattern.

## Faded exercise 2

### Faded — merge attention heads

Given `x` of shape `(b, h, s, d)` (the per-head attention output), complete `merge_heads(x)` to return `(b, s, h*d)` using one `rearrange`. The driver and check are provided; fill in the pattern string only.

**Fill in:** the rearrange pattern that reorders to (b, s, h, d) and composes the trailing (h d) into one axis.

In [ ]:
def merge_heads(x: Tensor) -> Tensor:
    raise NotImplementedError()  # TODO: return rearrange(x, <pattern>) giving shape (b, s, h*d)


np.random.seed(0)
t.manual_seed(0)
x = t.randn(2, 4, 7, 8)  # b=2, h=4, s=7, d=8
out = merge_heads(x)
print('output shape:', tuple(out.shape))


def _test():
    t.manual_seed(7)
    b, h, s, d = 2, 4, 7, 8
    x = t.randn(b, h, s, d)
    got = merge_heads(x)
    expected = x.permute(0, 2, 1, 3).reshape(b, s, h * d)
    assert got.shape == (b, s, h * d), f'wrong shape: {tuple(got.shape)}'
    assert t.equal(got, expected), 'values do not match permute+reshape reference'


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',  # single-exercise standalone — neutral signal
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def merge_heads(x: Tensor) -> Tensor:
    return rearrange(x, 'b h s d -> b s (h d)')


np.random.seed(0)
t.manual_seed(0)
x = t.randn(2, 4, 7, 8)  # b=2, h=4, s=7, d=8
out = merge_heads(x)
print('output shape:', tuple(out.shape))
```
</details>